In [2]:
# Walk up to repo root (identified by README.md)
.root <- getwd()
while (!file.exists(file.path(.root, 'README.md')) && dirname(.root) != .root)
  .root <- dirname(.root)
setwd(.root)
rm(.root)

suppressPackageStartupMessages({
  library(tidyverse)
  library(arrow)
  library(ggthemes)
  library(scales)
  library(patchwork)
})

options(repr.plot.width = 14, repr.plot.height = 8)

BASE_THEME <- theme_fivethirtyeight() +
  theme(
    axis.title        = element_text(size = 14),
    axis.text         = element_text(size = 13),
    panel.background  = element_rect(fill = "white", color = NA),
    plot.background   = element_rect(fill = "white", color = NA),
    legend.background = element_rect(fill = "white", color = NA),
    legend.text       = element_text(size = 12),
    legend.title      = element_text(size = 13),
    strip.text        = element_text(size = 14, face = "bold"),
    plot.title        = element_text(size = 20, face = "bold"),
    plot.subtitle     = element_text(size = 13)
  )
theme_set(BASE_THEME)

COMMIT_MS  <- 150

PITCH_NAMES <- c(
  FF = "4-Seam",    SI = "Sinker",    FC = "Cutter",
  CH = "Changeup",  FS = "Splitter",
  SL = "Slider",    CU = "Curveball", ST = "Sweeper",   KC = "Knuckle-curve"
)

PITCH_COLORS <- c(
  FF = "#d62728", SI = "#ff7f0e", FC = "#e377c2",
  CH = "#2ca02c", FS = "#17becf",
  SL = "#bcbd22", CU = "#1f77b4", ST = "#9467bd", KC = "#8c564b"
)


Warning message:
"il pacchetto 'arrow' `e stato creato con R versione 4.4.3"


In [3]:
pc <- read_parquet("data/swings_precommit.parquet") |>
  select(
    game_pk, at_bat_number, pitch_number,
    pc125_dev_total, pc150_dev_total, pc175_dev_total, pc200_dev_total,
    pc150_dev_x,     pc150_dev_z,
    pc150_emp_dev_x, pc150_emp_dev_z,
    pfx_x,           pfx_z
  )

sw <- read_csv("data/swings_2023_2025.csv", show_col_types = FALSE)

df <- sw |>
  left_join(pc, by = c("game_pk", "at_bat_number", "pitch_number")) |>
  filter(pitch_type %in% names(PITCH_NAMES)) |>
  mutate(
    pitch_name    = factor(PITCH_NAMES[pitch_type], levels = unname(PITCH_NAMES)),
    emp_dev_total = sqrt(pc150_emp_dev_x^2 + pc150_emp_dev_z^2),
    lin_dev_total = sqrt(pc150_dev_x^2 + pc150_dev_z^2)
  )

cat(sprintf(
  "Loaded %s swings | Years: %s | Bat-tracked: %.1f%%\n",
  formatC(nrow(df), format = "d", big.mark = ","),
  paste(sort(unique(df$game_year)), collapse = ", "),
  mean(!is.na(df$bat_speed)) * 100
))


ERROR: Error: IOError: Failed to open local file 'data/swings_precommit.parquet'. Detail: [errno 2] No such file or directory


## Data Coverage

In [ ]:
# ── Bat tracking coverage by year ─────────────────────────────────────────────
coverage <- df |>
  group_by(game_year) |>
  summarise(
    n_swings    = n(),
    n_tracked   = sum(!is.na(bat_speed)),
    pct_tracked = mean(!is.na(bat_speed)),
    .groups     = "drop"
  )

p1 <- ggplot(coverage, aes(x = factor(game_year), y = pct_tracked)) +
  geom_col(fill = "#3498db", width = 0.55) +
  geom_text(
    aes(label = sprintf("%s swings\n%.0f%% tracked",
                        formatC(n_swings, format = "d", big.mark = ","),
                        pct_tracked * 100)),
    vjust = -0.25, size = 4.5, lineheight = 1.1
  ) +
  scale_y_continuous(labels = percent_format(), limits = c(0, 1.15)) +
  labs(title = "Bat Tracking Coverage",
       subtitle = "Hawk-Eye bat tracking rolled out mid-2023",
       x = "Season", y = "% Swings with Bat Tracking")

# ── Pitch type distribution ────────────────────────────────────────────────────
pitch_dist <- df |>
  count(pitch_type) |>
  mutate(pitch_name = PITCH_NAMES[pitch_type]) |>
  arrange(desc(n))

p2 <- ggplot(pitch_dist,
             aes(x = reorder(pitch_name, n), y = n, fill = pitch_type)) +
  geom_col(width = 0.65) +
  geom_text(aes(label = formatC(n, format = "d", big.mark = ",")),
            hjust = -0.1, size = 4.3) +
  scale_fill_manual(values = PITCH_COLORS, guide = "none") +
  scale_y_continuous(expand = expansion(mult = c(0, 0.18)),
                     labels = comma_format()) +
  coord_flip() +
  labs(title = "Swings by Pitch Type",
       subtitle = "2023–2025",
       x = NULL, y = "Number of Swings")

p1 + p2 + plot_annotation(
  title = "Data Coverage",
  theme = theme(plot.title = element_text(size = 22, face = "bold"))
)


## Post-Commit Deviation Distributions

In [ ]:
# ── Post-commit deviation distributions by pitch type ─────────────────────────
df |>
  filter(!is.na(emp_dev_total)) |>
  ggplot(aes(x = emp_dev_total, fill = pitch_type, color = pitch_type)) +
  geom_density(alpha = 0.25, linewidth = 0.9) +
  facet_wrap(~ pitch_name, nrow = 3, scales = "free_y") +
  scale_fill_manual(values  = PITCH_COLORS, guide = "none") +
  scale_color_manual(values = PITCH_COLORS, guide = "none") +
  coord_cartesian(xlim = c(0, 0.75)) +
  labs(
    title    = "Post-Commit Deviation Distribution by Pitch Type",
    subtitle = "Empirical baseline (KNN) | pc150_emp_dev_total (ft) | right tail = more surprise",
    x        = "Empirical post-commit deviation — total (ft)",
    y        = "Density"
  )


## Empirical vs. Linear Baseline

In [ ]:
# ── Why not just use pfx? ─────────────────────────────────────────────────────
# Sample 30k rows for scatter readability
set.seed(42)
samp <- df |>
  filter(!is.na(pfx_z), !is.na(pc150_dev_z), !is.na(pc150_emp_dev_z)) |>
  slice_sample(n = min(30000, n()))

r_lin <- cor(samp$pfx_z, samp$pc150_dev_z,     use = "complete.obs")
r_emp <- cor(samp$pfx_z, samp$pc150_emp_dev_z, use = "complete.obs")

p1 <- ggplot(samp, aes(x = pfx_z, y = pc150_dev_z, color = pitch_type)) +
  geom_point(alpha = 0.07, size = 0.7, show.legend = FALSE) +
  scale_color_manual(values = PITCH_COLORS) +
  labs(
    title    = sprintf("Linear deviation vs pfx_z  (r = %.3f)", r_lin),
    subtitle = "Nearly 1:1 — lin_dev is pfx scaled by (tau/t_plate)²",
    x        = "pfx_z  (induced vertical break, ft)",
    y        = "Linear dev_z  (ft)"
  )

p2 <- ggplot(samp, aes(x = pfx_z, y = pc150_emp_dev_z, color = pitch_type)) +
  geom_point(alpha = 0.07, size = 0.7) +
  scale_color_manual(values = PITCH_COLORS, name = "Pitch") +
  labs(
    title    = sprintf("Empirical deviation vs pfx_z  (r = %.3f)", r_emp),
    subtitle = "Breaks the correlation — captures actual deception beyond raw movement",
    x        = "pfx_z  (induced vertical break, ft)",
    y        = "Empirical emp_dev_z  (ft)"
  )

p1 + p2 + plot_annotation(
  title    = "Why the Empirical Baseline, Not pfx?",
  subtitle = "A curveball that tips itself off at commit gets small emp_dev even with big pfx",
  theme    = theme(
    plot.title    = element_text(size = 20, face = "bold"),
    plot.subtitle = element_text(size = 13)
  )
)


## Deception Ratio by Pitch Type

In [ ]:
# ── Deception ratio: how surprising is each pitch type relative to its raw break? ─
pitch_dev <- df |>
  filter(!is.na(emp_dev_total), !is.na(lin_dev_total), lin_dev_total > 0.01) |>
  group_by(pitch_type) |>
  summarise(
    n             = n(),
    mean_emp_dev  = mean(emp_dev_total),
    mean_lin_dev  = mean(lin_dev_total),
    emp_lin_ratio = mean_emp_dev / mean_lin_dev,
    .groups       = "drop"
  ) |>
  filter(n >= 10000) |>
  mutate(pitch_name = PITCH_NAMES[pitch_type])

ggplot(pitch_dev,
       aes(x = reorder(pitch_name, emp_lin_ratio),
           y = emp_lin_ratio, fill = pitch_type)) +
  geom_col(width = 0.65) +
  geom_hline(yintercept = 1, linetype = "dashed", color = "grey40", linewidth = 0.8) +
  geom_text(aes(label = sprintf("%.2fx", emp_lin_ratio)),
            hjust = -0.15, size = 4.8, fontface = "bold") +
  scale_fill_manual(values = PITCH_COLORS, guide = "none") +
  scale_y_continuous(expand = expansion(mult = c(0, 0.18))) +
  coord_flip() +
  labs(
    title    = "Deception Ratio by Pitch Type",
    subtitle = "emp_dev / lin_dev  |  > 1× = pitch arrives more surprisingly than raw movement predicts\nDashed line = pitch is exactly as surprising as its movement suggests",
    x        = NULL,
    y        = "Empirical ÷ Linear deviation (ratio)"
  )


## Commit-Time Sensitivity

In [ ]:
# ── Commit-time sensitivity: does the 150ms choice matter? ───────────────────
sensitivity <- df |>
  filter(!is.na(pc125_dev_total), !is.na(pc150_dev_total),
         !is.na(pc175_dev_total), !is.na(pc200_dev_total)) |>
  select(pitch_type, starts_with("pc") & ends_with("dev_total")) |>
  pivot_longer(
    cols      = starts_with("pc") & ends_with("dev_total"),
    names_to  = "col",
    values_to = "dev_total"
  ) |>
  mutate(commit_ms = as.integer(str_extract(col, "(?<=pc)\\d+"))) |>
  filter(commit_ms %in% c(125, 150, 175, 200))

sens_agg <- sensitivity |>
  group_by(pitch_type, commit_ms) |>
  summarise(mean_dev = mean(dev_total, na.rm = TRUE), .groups = "drop") |>
  mutate(pitch_name = PITCH_NAMES[pitch_type])

ggplot(sens_agg,
       aes(x = commit_ms, y = mean_dev,
           color = pitch_name, group = pitch_name)) +
  geom_line(linewidth = 1.1) +
  geom_point(size = 3.5) +
  geom_vline(xintercept = 150, linetype = "dashed",
             color = "grey40", linewidth = 0.8) +
  annotate("text", x = 152, y = max(sens_agg$mean_dev) * 0.97,
           label = "Primary\n150 ms", hjust = 0, size = 4, color = "grey40") +
  scale_color_manual(
    values = setNames(PITCH_COLORS[names(PITCH_NAMES)], unname(PITCH_NAMES)),
    name   = "Pitch type"
  ) +
  scale_x_continuous(breaks = c(125, 150, 175, 200),
                     labels = c("125 ms", "150 ms", "175 ms", "200 ms")) +
  labs(
    title    = "Commit-Time Sensitivity",
    subtitle = "Mean linear post-commit deviation at 125 / 150 / 175 / 200 ms\nRelative ordering between pitch types is stable — the 150 ms choice is not load-bearing",
    x        = "Commit time before contact",
    y        = "Mean post-commit deviation (ft)"
  )


## Late Movement by Count

In [ ]:
# ── Late movement magnitude by count ─────────────────────────────────────────
count_dev <- df |>
  filter(!is.na(emp_dev_total), balls <= 3, strikes <= 2) |>
  group_by(balls, strikes) |>
  summarise(
    n            = n(),
    mean_emp_dev = mean(emp_dev_total),
    .groups      = "drop"
  ) |>
  mutate(
    label = sprintf("%.3f ft\n(n=%s)",
                    mean_emp_dev,
                    formatC(n, format = "d", big.mark = ","))
  )

ggplot(count_dev,
       aes(x = factor(strikes), y = factor(balls), fill = mean_emp_dev)) +
  geom_tile(color = "white", linewidth = 1) +
  geom_text(aes(label = label),
            size = 5, color = "white", fontface = "bold", lineheight = 1.2) +
  scale_fill_gradient(
    low  = "#3498db", high = "#c0392b",
    name = "Mean emp\ndev_total (ft)"
  ) +
  scale_x_discrete(labels = c("0 strikes", "1 strike", "2 strikes")) +
  scale_y_discrete(labels = c("0 balls", "1 ball", "2 balls", "3 balls")) +
  labs(
    title    = "Late Movement Magnitude by Count",
    subtitle = "Mean empirical post-commit deviation — do pitchers throw more deceptive stuff in pitcher counts?",
    x        = "Strikes",
    y        = "Balls"
  )
